# Collect data-related job listings

Python 3 replacement for the 2018 Careerjet notebook.

Careerjet now needs a publisher key. This notebook reads **public JSON APIs**
(Jobicy by default) with timeouts and a project User-Agent.

If the network call fails, load `data/texas_jobs_2018.csv` instead so the
rest of the analysis still runs.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "job_scraping").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from job_scraping.http import FetchError
from job_scraping.skills import analyze_jobs
from job_scraping.sources import collect_jobs

pd.set_option("display.max_colwidth", 80)


In [ ]:
HISTORICAL = ROOT / "data" / "texas_jobs_2018.csv"
OUTPUT = ROOT / "data" / "jobs.csv"

try:
    jobs = collect_jobs(
        source="jobicy",
        queries=["data scientist", "data analyst"],
        limit=40,
    )
    raw = pd.DataFrame(
        [{**job.to_public_row(), "description_text": job.description} for job in jobs]
    )
    print(f"Fetched {len(raw)} public listings")
except FetchError as exc:
    print(f"Live collect failed ({exc}). Using the 2018 Texas snapshot.")
    raw = pd.read_csv(HISTORICAL)

raw.head()


In [ ]:
analyzed = analyze_jobs(raw)
analyzed = analyzed.drop(columns=["description_text"], errors="ignore")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
# Skill flags only — no full job-ad text.
analyzed.to_csv(OUTPUT, index=False)
print(f"Wrote {OUTPUT} ({len(analyzed)} rows)")
analyzed[["title", "company", "locations", "sql", "python", "Bachelor", "Master"]].head()


The 2018 notebooks also tried to pull the *actual* employer URL out of a Careerjet interstitial page. That HTML is gone. Public APIs already return a listing URL in the `url` column.
